In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [2]:
data = pd.read_csv("dataset_SO.csv")
data.head()



,id,serie,nom_spe1,nom_spe2,note_maths,note_francais,note_anglais,note_philo,note_hg,note_spe1,...,score_S,score_E,score_C,code_holland,stress,social,autonomie,adaptabilite,branche,categorie
0,1,L1,LV2,—,11.71,14.19,14.99,16.39,14.44,13.33,...,4.66,4.79,3.78,ESC,3.49,4.65,3.68,3.97,Sciences Politiques,Droit & Sciences Politiques
1,2,S1,Physique-Chimie,—,13.56,14.46,15.57,12.72,13.78,17.23,...,4.24,3.11,3.66,ISR,4.54,4.49,3.73,3.97,Médecine,Sciences de la Santé
2,3,L1,LV2,—,11.01,17.88,18.04,16.04,13.55,15.41,...,4.50,4.46,3.70,SEC,3.85,3.98,3.67,4.25,Sciences Politiques,Droit & Sciences Politiques
3,4,L2,LV2,—,8.61,17.35,14.43,13.62,17.03,16.96,...,4.43,4.69,3.35,ESI,3.47,4.55,3.76,3.92,Sciences Politiques,Droit & Sciences Politiques
4,5,L2,LV2,—,8.05,13.03,12.87,14.79,15.17,12.72,...,4.82,2.99,2.44,SIA,3.35,4.47,3.38,5.00,Sociologie,Sciences Humaines & Sociales


In [3]:
data.shape

(6830, 27)

In [4]:
data.dtypes

id                 int64
serie                str
nom_spe1             str
nom_spe2             str
note_maths       float64
note_francais    float64
note_anglais     float64
note_philo       float64
note_hg          float64
note_spe1        float64
note_spe2        float64
moy_sciences     float64
moy_lettres      float64
moy_globale      float64
score_R          float64
score_I          float64
score_A          float64
score_S          float64
score_E          float64
score_C          float64
code_holland         str
stress           float64
social           float64
autonomie        float64
adaptabilite     float64
branche              str
categorie            str
dtype: object

In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 6830 entries, 0 to 6829
Data columns (total 27 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             6830 non-null   int64  
 1   serie          6830 non-null   str    
 2   nom_spe1       6830 non-null   str    
 3   nom_spe2       6830 non-null   str    
 4   note_maths     6830 non-null   float64
 5   note_francais  6830 non-null   float64
 6   note_anglais   6830 non-null   float64
 7   note_philo     6830 non-null   float64
 8   note_hg        6830 non-null   float64
 9   note_spe1      6830 non-null   float64
 10  note_spe2      6830 non-null   float64
 11  moy_sciences   6830 non-null   float64
 12  moy_lettres    6830 non-null   float64
 13  moy_globale    6830 non-null   float64
 14  score_R        6830 non-null   float64
 15  score_I        6830 non-null   float64
 16  score_A        6830 non-null   float64
 17  score_S        6830 non-null   float64
 18  score_E        6830

In [6]:
print (data.isnull().sum())

id               0
serie            0
nom_spe1         0
nom_spe2         0
note_maths       0
note_francais    0
note_anglais     0
note_philo       0
note_hg          0
note_spe1        0
note_spe2        0
moy_sciences     0
moy_lettres      0
moy_globale      0
score_R          0
score_I          0
score_A          0
score_S          0
score_E          0
score_C          0
code_holland     0
stress           0
social           0
autonomie        0
adaptabilite     0
branche          0
categorie        0
dtype: int64


Suppression des colonnes inutiles

In [8]:
data = data.drop(columns=['id'])

Séparation de la variable cible 

In [9]:
X = data.drop(columns=['categorie'])
y= data['categorie']

Les modèles ne comprenant pas le texte, il va falloir encoder les variables catégorielles, exemple : serie, nom_spe1 etc...
tout d'abord il faut identifier ces variables, ensuite nous allons gérer l'encodage avec OneHotEncoder.

In [13]:
cat_cols= X.select_dtypes(include=['object','str']).columns
num_cols= X.select_dtypes(include=['Int64', 'float64']).columns

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

Nous allons effetuer un split de 70/15/15, train/val/test.
Nous allons fire 2 splits, dans un premier temps séparer le train du reste, ensuite séparer val et test.

In [18]:
X_train, X_temp, y_train, y_temp = train_test_split (
    X,
    y,
    test_size=0.30, 
    random_state=42,
    stratify=y

)

In [19]:
X_val, X_test, y_val, y_test= train_test_split (
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

Application du prétraitement

In [21]:
#fit sur train
X_train_processed= preprocessor.fit_transform(X_train)

#transform seulement sur val et test
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)


Nous allons utilisé SMOTE pour corriger le déséquilibre des classes dans le dataset. Cette méthode génère artificiellement de nouveaux exemples pour les classes minoritaires afin d’éviter que le modèle soit biaisé vers les classes majoritaires.

Vérification de l'équilibrage avant smote

In [26]:
print(y_train.value_counts())

categorie
Sciences & Technologies            1106
Sciences de la Santé                840
Sciences Économiques & Gestion      518
Multimédia & Création Numérique     490
Agriculture & Environnement         455
Sciences Humaines & Sociales        385
Droit & Sciences Politiques         364
Lettres & Communication             343
Arts & Culture                      280
Name: count, dtype: int64


In [25]:
smote= SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample (
    X_train_processed,
    y_train
)

Vérification après smote

In [27]:
print(pd.Series(y_train_smote).value_counts())

categorie
Droit & Sciences Politiques        1106
Lettres & Communication            1106
Sciences de la Santé               1106
Sciences & Technologies            1106
Arts & Culture                     1106
Agriculture & Environnement        1106
Sciences Humaines & Sociales       1106
Sciences Économiques & Gestion     1106
Multimédia & Création Numérique    1106
Name: count, dtype: int64
